In [2]:
# ============================================================
# Startup Cell: Mount Drive + Load Config + Verify Inputs
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys

# ------------------------------------------------------------
# Add project source directory to Python path
# ------------------------------------------------------------
SRC_DIR = "/content/drive/MyDrive/DIP_Project/src"

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# ------------------------------------------------------------
# Import project configuration
# ------------------------------------------------------------
from project_config import *

print("Project configuration loaded.")
print(f"BASE_DIR:     {BASE_DIR}")
print(f"METADATA_DIR: {METADATA_DIR}")
print(f"MODELS_DIR:   {MODELS_DIR}")

# ------------------------------------------------------------
# Verify required input files
# ------------------------------------------------------------
print("\nVerifying required normalized feature files...")

required_files = [
    os.path.join(METADATA_DIR, TRAIN_NORMALIZED_FILENAME),
    os.path.join(METADATA_DIR, TEST_NORMALIZED_FILENAME),
]

all_files_present = True

for file_path in required_files:
    if os.path.exists(file_path):
        print(f"{os.path.basename(file_path):50s} OK")
    else:
        print(f"{os.path.basename(file_path):50s} MISSING")
        all_files_present = False

# ------------------------------------------------------------
# Verify / create output directories
# ------------------------------------------------------------
print("\nVerifying output directories...")

required_dirs = [
    MODELS_DIR,
]

for dir_path in required_dirs:
    if not os.path.exists(dir_path):
        os.makedirs(dir_path, exist_ok=True)
        print(f"Created: {dir_path}")
    else:
        print(f"Exists:  {dir_path}")

# ------------------------------------------------------------
# Display key runtime parameters (sanity check)
# ------------------------------------------------------------
print("\nKey configuration values:")
print(f"Number of features: {NUM_FEATURES}")
print(f"Train/Test split:   {TRAIN_RATIO:.2f} / {TEST_RATIO:.2f}")
print(f"K-Folds (CV):       {K_FOLDS}")
print(f"Random seed:        {RANDOM_SEED}")

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------
if all_files_present:
    print("\nAll required files are present. Ready to proceed.")
else:
    print("\nERROR: Missing required input files. Check paths before continuing.")



Mounted at /content/drive
Project configuration loaded.
BASE_DIR:     /content/drive/MyDrive/DIP_Project
METADATA_DIR: /content/drive/MyDrive/DIP_Project/data/metadata
MODELS_DIR:   /content/drive/MyDrive/DIP_Project/models

Verifying required normalized feature files...
train_feature_vectors_normalized.csv               OK
test_feature_vectors_normalized.csv                OK

Verifying output directories...
Exists:  /content/drive/MyDrive/DIP_Project/models

Key configuration values:
Number of features: 25
Train/Test split:   0.80 / 0.20
K-Folds (CV):       5
Random seed:        42

All required files are present. Ready to proceed.


In [3]:
# ------------------------------------------------------------
# What the notebook does:
# ------------------------------------------------------------
#
#   Startup Cell:
#     Mount Google Drive, import project_config.py, and verify
#     that required normalized train/test feature-vector CSV
#     files exist.
#
#   Cell 1:
#     Import required libraries for model training,
#     cross-validation, evaluation metrics, and model saving.
#
#   Cell 2:
#     Load normalized training and test datasets and display
#     basic shape and structure information.
#
#   Cell 3:
#     Define metadata columns and extract feature matrices (X)
#     and labels (y) for training and test datasets.
#
#   Cell 4:
#     Validate inputs:
#       - confirm feature dimensions
#       - confirm no missing values
#       - confirm label consistency (ai vs real)
#
#   Cell 5:
#     Define classifier configurations:
#       - Multi-Layer Perceptron (MLP)
#       - RBF Support Vector Machine (RBF SVM)
#
#   Cell 6:
#     Set up cross-validation framework (Stratified K-Fold)
#     and define evaluation metrics:
#       - accuracy
#       - precision
#       - recall
#       - F1-score
#       - ROC-AUC
#
#   Cell 7:
#     Perform cross-validation for both classifiers and collect
#     performance metrics across folds.
#
#   Cell 8:
#     Summarize and compare classifier performance:
#       - mean and standard deviation of each metric
#       - rank models based on ROC-AUC
#
#   Cell 9:
#     Select the best-performing classifier based on
#     cross-validation results.
#
#   Cell 10:
#     Retrain the selected classifier using the full training
#     dataset.
#
#   Cell 11:
#     Save final trained model to .pkl file.
#
#   Cell 12:
#     Save cross-validation results and best model configuration:
#       - cross_validation_results.csv
#       - best_model_config.json
#
#   Cell 13 (optional but recommended):
#     Display final summary:
#       - selected model name
#       - key hyperparameters
#       - cross-validation performance metrics
#
# ------------------------------------------------------------
# Notes:
# ------------------------------------------------------------
# - Only the training dataset is used for model selection.
# - The test dataset remains untouched for final evaluation
#   in Notebook 09.
# - Cross-validation ensures robust comparison between MLP
#   and RBF SVM without introducing data leakage.
# - ROC-AUC is the primary selection metric due to its
#   threshold-independent evaluation of classifier performance.
# ------------------------------------------------------------



In [4]:
# ============================================================
# Cell 1: Imports
# ============================================================

import os
import json
import pickle
import warnings

import numpy as np
import pandas as pd

import time

from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

print("Imports loaded successfully.")


Imports loaded successfully.


In [5]:
# ============================================================
# Cell 2: Load Normalized Training and Test Data
# ============================================================

train_csv_path = os.path.join(METADATA_DIR, TRAIN_NORMALIZED_FILENAME)
test_csv_path = os.path.join(METADATA_DIR, TEST_NORMALIZED_FILENAME)

df_train = pd.read_csv(train_csv_path)
df_test = pd.read_csv(test_csv_path)

print("Normalized training and test feature tables loaded successfully.\n")

print(f"Training CSV: {train_csv_path}")
print(f"Test CSV:     {test_csv_path}\n")

print(f"df_train shape: {df_train.shape}")
print(f"df_test shape:  {df_test.shape}\n")

print("First 5 rows of training table:")
display(df_train.head())



Normalized training and test feature tables loaded successfully.

Training CSV: /content/drive/MyDrive/DIP_Project/data/metadata/train_feature_vectors_normalized.csv
Test CSV:     /content/drive/MyDrive/DIP_Project/data/metadata/test_feature_vectors_normalized.csv

df_train shape: (14400, 29)
df_test shape:  (3600, 29)

First 5 rows of training table:


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Patch Variance Std,Noise Residual Energy,Low Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train,0.973214,1.208264,0.913526,0.776979,0.735101,-0.207859,...,1.019527,1.071297,-0.189219,0.417247,0.448644,0.439427,-0.549996,-0.084929,0.290945,-1.254194
1,rl_coco_001397.png,rl,MS_COCO_2017,train,-0.450913,-0.225225,0.641767,-0.488332,-0.447341,-0.246587,...,0.509586,-0.330229,-0.264290,0.144170,-1.327122,-1.325683,1.397134,0.856623,0.808074,0.071118
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train,0.605933,0.723094,0.717432,0.610900,0.160581,-0.468100,...,0.235488,0.851489,-0.544033,0.805334,-0.343146,-0.332461,-0.549996,-0.217414,0.432281,-1.341491
3,rl_coco_000800.png,rl,MS_COCO_2017,train,0.021388,0.357107,0.528482,-0.077406,0.331237,1.075664,...,0.395378,0.188882,0.518813,-0.349898,1.963337,1.956027,-0.549996,-0.448685,-0.628062,-1.111831
4,ai_mj_002892.png,ai,Midjourney,train,-0.212459,-0.661686,-0.209270,0.288772,-0.061089,-0.293840,...,-0.373574,-0.281104,0.607047,-0.556607,-0.011009,-0.013482,-0.549996,-0.185376,-0.470299,0.325939


In [7]:
# ============================================================
# Cell 3: Sanity Checks + Separate Features and Labels
# ============================================================

# ------------------------------------------------------------
# Expected metadata columns
# ------------------------------------------------------------
expected_metadata_columns = METADATA_COLUMNS.copy()

# ------------------------------------------------------------
# Verify required metadata columns
# ------------------------------------------------------------
missing_train_metadata = [col for col in expected_metadata_columns if col not in df_train.columns]
missing_test_metadata = [col for col in expected_metadata_columns if col not in df_test.columns]

if missing_train_metadata:
    raise ValueError(f"Training table is missing metadata columns: {missing_train_metadata}")

if missing_test_metadata:
    raise ValueError(f"Test table is missing metadata columns: {missing_test_metadata}")

print("Required metadata columns verified.")

# ------------------------------------------------------------
# Verify feature columns
# ------------------------------------------------------------
feature_columns = [col for col in df_train.columns if col not in METADATA_COLUMNS]
test_feature_columns = [col for col in df_test.columns if col not in METADATA_COLUMNS]

if len(feature_columns) != NUM_FEATURES:
    raise ValueError(
        f"Expected {NUM_FEATURES} feature columns, but found {len(feature_columns)}."
    )

print(f"Feature count verified: {len(feature_columns)}")

if feature_columns != test_feature_columns:
    raise ValueError("Training and test feature columns do not match exactly.")

print("Training and test feature columns match.")

# ------------------------------------------------------------
# Verify no missing values
# ------------------------------------------------------------
if df_train.isnull().sum().sum() != 0:
    raise ValueError("Training table contains missing values.")

if df_test.isnull().sum().sum() != 0:
    raise ValueError("Test table contains missing values.")

print("No missing values detected.")

# ------------------------------------------------------------
# Separate features and labels
# ------------------------------------------------------------
X_train = df_train[feature_columns].copy()
X_test = df_test[feature_columns].copy()

y_train = df_train["class_label"].copy()
y_test = df_test["class_label"].copy()

# ------------------------------------------------------------
# Verify label consistency
# ------------------------------------------------------------
expected_labels = {AI_LABEL, REAL_LABEL}

train_labels_found = set(y_train.unique())
test_labels_found = set(y_test.unique())

if not train_labels_found.issubset(expected_labels):
    raise ValueError(f"Unexpected labels found in training set: {train_labels_found}")

if not test_labels_found.issubset(expected_labels):
    raise ValueError(f"Unexpected labels found in test set: {test_labels_found}")

print(f"Label values verified: {sorted(expected_labels)}")

# ------------------------------------------------------------
# Display class distributions
# ------------------------------------------------------------
print("\nTraining class counts:")
print(y_train.value_counts())

print("\nTest class counts:")
print(y_test.value_counts())

# ------------------------------------------------------------
# Display matrix/vector shapes
# ------------------------------------------------------------
print("\nFeature matrix shapes:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

print("\nLabel vector shapes:")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

print("\nSanity checks passed successfully.")



Required metadata columns verified.
Feature count verified: 25
Training and test feature columns match.
No missing values detected.
Label values verified: ['ai', 'rl']

Training class counts:
class_label
rl    7200
ai    7200
Name: count, dtype: int64

Test class counts:
class_label
rl    1800
ai    1800
Name: count, dtype: int64

Feature matrix shapes:
X_train: (14400, 25)
X_test:  (3600, 25)

Label vector shapes:
y_train: (14400,)
y_test:  (3600,)

Sanity checks passed successfully.


In [8]:
# ============================================================
# Cell 4: Define Candidate Classifier Configurations
# ============================================================

candidate_configs = [
    {
        "model_name": "MLP_128_64_32_alpha0.001",
        "model_type": "MLP",
        "params": {
            "hidden_layer_sizes": (128, 64, 32),
            "alpha": 0.001,
            "max_iter": 300,
            "random_state": RANDOM_SEED,
        },
    },
    {
        "model_name": "RBF_SVM_C100_gamma0.01",
        "model_type": "SVM",
        "params": {
            "C": 100.0,
            "gamma": 0.01,
            "kernel": "rbf",
            "probability": True,
            "random_state": RANDOM_SEED,
        },
    },
]

print("Candidate classifier configurations defined successfully.\n")
print(f"Number of candidate models: {len(candidate_configs)}\n")

for config in candidate_configs:
    print(f"{config['model_name']}:")
    for param, value in config["params"].items():
        print(f"  {param} = {value}")
    print()



Candidate classifier configurations defined successfully.

Number of candidate models: 2

MLP_128_64_32_alpha0.001:
  hidden_layer_sizes = (128, 64, 32)
  alpha = 0.001
  max_iter = 300
  random_state = 42

RBF_SVM_C100_gamma0.01:
  C = 100.0
  gamma = 0.01
  kernel = rbf
  probability = True
  random_state = 42



In [9]:
# ============================================================
# Cell 5: Cross-Validate Candidate Classifier Models
# ============================================================

import time

cv = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=CV_SHUFFLE,
    random_state=CV_RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc",
}

results = []

print("Evaluating candidate classifier models...\n")
print(f"Number of models: {len(candidate_configs)}")
print(f"Cross-validation folds: {K_FOLDS}\n")

total_start_time = time.time()

for i, config in enumerate(candidate_configs, start=1):
    model_start_time = time.time()

    print(f"[{i}/{len(candidate_configs)}] {config['model_name']}")

    # --------------------------------------------------------
    # Instantiate model
    # --------------------------------------------------------
    if config["model_type"] == "MLP":
        model = MLPClassifier(**config["params"])

    elif config["model_type"] == "SVM":
        model = SVC(**config["params"])

    else:
        raise ValueError(f"Unsupported model type: {config['model_type']}")

    # --------------------------------------------------------
    # Cross-validation
    # --------------------------------------------------------
    cv_results = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    # --------------------------------------------------------
    # Save summary row
    # --------------------------------------------------------
    row = {
        "model_name": config["model_name"],
        "model_type": config["model_type"],
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std": cv_results["test_accuracy"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "precision_std": cv_results["test_precision"].std(),
        "recall_mean": cv_results["test_recall"].mean(),
        "recall_std": cv_results["test_recall"].std(),
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_std": cv_results["test_f1"].std(),
        "roc_auc_mean": cv_results["test_roc_auc"].mean(),
        "roc_auc_std": cv_results["test_roc_auc"].std(),
    }

    for param_name, param_value in config["params"].items():
        row[param_name] = str(param_value)

    results.append(row)

    model_elapsed = time.time() - model_start_time
    total_elapsed = time.time() - total_start_time

    print(f"  Completed in {model_elapsed:.2f} seconds")
    print(f"  Total elapsed: {total_elapsed:.2f} seconds\n")

print("Cross-validation complete.")
print(f"Total runtime: {time.time() - total_start_time:.2f} seconds")



Evaluating candidate classifier models...

Number of models: 2
Cross-validation folds: 5

[1/2] MLP_128_64_32_alpha0.001
  Completed in 40.37 seconds
  Total elapsed: 40.37 seconds

[2/2] RBF_SVM_C100_gamma0.01
  Completed in 80.12 seconds
  Total elapsed: 120.49 seconds

Cross-validation complete.
Total runtime: 120.49 seconds


In [11]:
# ============================================================
# Cell 6: Summarize and Rank Candidate Model Results
# ============================================================

df_cv_results = pd.DataFrame(results)

# Sort by primary metric (ROC-AUC)
df_cv_results = df_cv_results.sort_values(
    by="roc_auc_mean",
    ascending=False
).reset_index(drop=True)

print("Candidate model comparison (sorted by ROC-AUC):\n")
display(df_cv_results)

# ------------------------------------------------------------
# Identify best model
# ------------------------------------------------------------
best_row = df_cv_results.iloc[0]

best_model_name = best_row["model_name"]
best_model_type = best_row["model_type"]

# Recover best parameters from candidate_configs
best_config = next(
    config for config in candidate_configs
    if config["model_name"] == best_model_name
)

best_params = best_config["params"]

print("Best configuration identified:\n")
print(f"Model: {best_model_name}")
print(f"Type:  {best_model_type}")

for key, value in best_params.items():
    print(f"  {key}: {value}")

Candidate model comparison (sorted by ROC-AUC):



,model_name,model_type,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,hidden_layer_sizes,alpha,max_iter,random_state,C,gamma,kernel,probability
0,RBF_SVM_C100_gamma0.01,SVM,0.798194,0.008415,0.798329,0.008319,0.798194,0.008415,0.798171,0.008434,0.879229,0.005399,NaN,NaN,NaN,42,100.0,0.01,rbf,True
1,MLP_128_64_32_alpha0.001,MLP,0.759097,0.011414,0.760215,0.010545,0.759097,0.011414,0.758819,0.011666,0.837371,0.010732,"(128, 64, 32)",0.001,300,42,NaN,NaN,NaN,NaN


Best configuration identified:

Model: RBF_SVM_C100_gamma0.01
Type:  SVM
  C: 100.0
  gamma: 0.01
  kernel: rbf
  probability: True
  random_state: 42


In [13]:
# ============================================================
# Cell 7: Train Final Models on Full Training Set
# ============================================================

trained_models = {}

print("Training final models on full training dataset...\n")

for config in candidate_configs:
    model_name = config["model_name"]
    model_type = config["model_type"]
    params = config["params"]

    print(f"Training: {model_name}")

    # --------------------------------------------------------
    # Instantiate model
    # --------------------------------------------------------
    if model_type == "MLP":
        model = MLPClassifier(**params)

    elif model_type == "SVM":
        model = SVC(**params)

    else:
        raise ValueError(f"Unsupported model type: {model_type}")

    # --------------------------------------------------------
    # Train model
    # --------------------------------------------------------
    model.fit(X_train, y_train)

    # Store trained model
    trained_models[model_name] = model

    print(f"  Completed: {model_name}\n")

print("All models trained successfully.")



Training final models on full training dataset...

Training: MLP_128_64_32_alpha0.001
  Completed: MLP_128_64_32_alpha0.001

Training: RBF_SVM_C100_gamma0.01
  Completed: RBF_SVM_C100_gamma0.01

All models trained successfully.


In [14]:
# ============================================================
# Cell 8: Save Trained Models
# ============================================================

saved_model_paths = {}

print("Saving trained models...\n")

for model_name, model in trained_models.items():

    # Create filename based on model name
    filename = f"{model_name}.pkl"
    model_path = os.path.join(MODELS_DIR, filename)

    # Save model
    with open(model_path, "wb") as f:
        pickle.dump(model, f)

    saved_model_paths[model_name] = model_path

    print(f"Saved: {model_name}")
    print(f"  Path: {model_path}\n")

print("All models saved successfully.")



Saving trained models...

Saved: MLP_128_64_32_alpha0.001
  Path: /content/drive/MyDrive/DIP_Project/models/MLP_128_64_32_alpha0.001.pkl

Saved: RBF_SVM_C100_gamma0.01
  Path: /content/drive/MyDrive/DIP_Project/models/RBF_SVM_C100_gamma0.01.pkl

All models saved successfully.


In [16]:
# ============================================================
# Cell 9: Save Cross-Validation Summary and Model Configs
# ============================================================

# ------------------------------------------------------------
# Define output paths
# ------------------------------------------------------------
cv_results_mlp_path = os.path.join(METADATA_DIR, "cross_validation_results_mlp.csv")
cv_results_rbf_path = os.path.join(METADATA_DIR, "cross_validation_results_rbf_svm.csv")

best_mlp_config_path = os.path.join(MODELS_DIR, "best_mlp_model_config.json")
best_rbf_config_path = os.path.join(MODELS_DIR, "best_rbf_svm_model_config.json")

all_configs_path = os.path.join(MODELS_DIR, "all_model_configs.json")

# ------------------------------------------------------------
# Build cross-validation summary tables by model
# ------------------------------------------------------------
df_cv_results_mlp = df_cv_results[
    df_cv_results["model_type"].str.upper() == "MLP"
].copy()

df_cv_results_rbf = df_cv_results[
    df_cv_results["model_type"].str.upper() == "SVM"
].copy()

# ------------------------------------------------------------
# Save cross-validation summary tables
# ------------------------------------------------------------
df_cv_results_mlp.to_csv(cv_results_mlp_path, index=False)
df_cv_results_rbf.to_csv(cv_results_rbf_path, index=False)

# ------------------------------------------------------------
# Extract config dictionaries for saved finalist models
# ------------------------------------------------------------
mlp_config = next(
    config for config in candidate_configs
    if config["model_name"] == "MLP_128_64_32_alpha0.001"
)

rbf_config = next(
    config for config in candidate_configs
    if config["model_name"] == "RBF_SVM_C100_gamma0.01"
)

# ------------------------------------------------------------
# Extract matching CV rows
# ------------------------------------------------------------
best_mlp_row = df_cv_results[
    df_cv_results["model_name"] == "MLP_128_64_32_alpha0.001"
].iloc[0]

best_rbf_row = df_cv_results[
    df_cv_results["model_name"] == "RBF_SVM_C100_gamma0.01"
].iloc[0]

# ------------------------------------------------------------
# Save best MLP model configuration
# ------------------------------------------------------------
best_mlp_model_config = {
    "model_name": mlp_config["model_name"],
    "model_type": mlp_config["model_type"],
    "hyperparameters": mlp_config["params"],
    "num_features": NUM_FEATURES,
    "k_folds": K_FOLDS,
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "selected_metric": "roc_auc",
    "roc_auc_mean": float(best_mlp_row["roc_auc_mean"]),
}

with open(best_mlp_config_path, "w") as f:
    json.dump(best_mlp_model_config, f, indent=4)

# ------------------------------------------------------------
# Save best RBF SVM model configuration
# ------------------------------------------------------------
best_rbf_model_config = {
    "model_name": rbf_config["model_name"],
    "model_type": rbf_config["model_type"],
    "hyperparameters": rbf_config["params"],
    "num_features": NUM_FEATURES,
    "k_folds": K_FOLDS,
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "selected_metric": "roc_auc",
    "roc_auc_mean": float(best_rbf_row["roc_auc_mean"]),
}

with open(best_rbf_config_path, "w") as f:
    json.dump(best_rbf_model_config, f, indent=4)

# ------------------------------------------------------------
# Save ALL model configurations
# ------------------------------------------------------------
with open(all_configs_path, "w") as f:
    json.dump(candidate_configs, f, indent=4)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------
print("Cross-validation summaries saved successfully.")
print(f"MLP CSV:      {cv_results_mlp_path}")
print(f"RBF SVM CSV:  {cv_results_rbf_path}")

print("\nBest model configurations saved successfully.")
print(f"MLP JSON:      {best_mlp_config_path}")
print(f"RBF SVM JSON:  {best_rbf_config_path}")

print("\nAll model configurations saved successfully.")
print(f"JSON: {all_configs_path}")



Cross-validation summaries saved successfully.
MLP CSV:      /content/drive/MyDrive/DIP_Project/data/metadata/cross_validation_results_mlp.csv
RBF SVM CSV:  /content/drive/MyDrive/DIP_Project/data/metadata/cross_validation_results_rbf_svm.csv

Best model configurations saved successfully.
MLP JSON:      /content/drive/MyDrive/DIP_Project/models/best_mlp_model_config.json
RBF SVM JSON:  /content/drive/MyDrive/DIP_Project/models/best_rbf_svm_model_config.json

All model configurations saved successfully.
JSON: /content/drive/MyDrive/DIP_Project/models/all_model_configs.json
